In [ ]:
from pathlib import Path
import os
import sys
import json

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.common.nuscenes_utils import (
    get_scene_contents,
    get_sample_contents,
    get_sample_window_contents
)
from src.common.geometry.pointcloud import (
    transform_lidar_to_ego,
    transform_ego_to_global
)
from src.common.geometry.depth import transform_cam_to_ego, depth_map_to_point_cloud
from src.common.geometry.transform import make_transform, invert_transform
from src.common.visualize.pointcloud import plot_pointcloud
from src.common.visualize.depth import plot_depth_with_original_image, plot_pseudo_lidar_with_ground_truth
from src.common.frame_ops import create_sliding_windows

from src.video_depth_anything.inference import load_model

# Resolve paths relative to this notebook directory
CHECKPOINTS_DIR = ROOT / "Video-Depth-Anything" / "checkpoints"
ENCODER_NAME = "vitl" # encoder name, can be "vits", "vitb" or "vitl"
METRIC = True
INPUT_SIZE = 518 # input size for the model
FP32 = True # If True, the model will run in fp32 mode, otherwise it will run in fp16 mode. Note that fp16 mode is faster but less accurate.
device = "cuda" if torch.cuda.is_available() else "cpu"

# Create the model and load the weights
model = load_model(ENCODER_NAME, checkpoint_dir=CHECKPOINTS_DIR, metric=METRIC, device=device)

# Load nuScenes dataset
NUSCENES_ROOT = Path.cwd().parent / "data/nuscenes"
NUSCENES_VERSION = "v1.0-trainval"
with open(NUSCENES_ROOT / NUSCENES_VERSION / "scene.json") as f:
    scenes = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample.json") as f:
    samples_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_data.json") as f:
    sample_data_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "ego_pose.json") as f:
    ego_poses_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "calibrated_sensor.json") as f:
    calibrated_sensors_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sensor.json") as f:
    sensors = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "sample_annotation.json") as f:
    sample_annotations_all = json.load(f)
with open(NUSCENES_ROOT / NUSCENES_VERSION / "instance.json") as f:
    instances_all = json.load(f)

# Create hash maps for token lookup
sensors = {sensor["token"]: sensor for sensor in sensors}
sensor_lookup = {sensor["channel"]: sensor["token"] for sensor in sensors.values()}

print(f"sensor_channels: {[sensor['channel'] for sensor in sensors.values()]}")
print(f"scene_names: {[scene['name'] for scene in scenes]}")

In [ ]:
# Pose-conditioned depth estimation with sliding window
SCENE_NAME = "scene-0061"
CAMERA_CHANNEL = "CAM_FRONT"
LIDAR_CHANNEL = "LIDAR_TOP"

WINDOW_SIZE = 13  # Number of frames in the sliding window
STRIDE = 9  # Stride for sliding window

# Get the scene contents for the selected scene
scene = next(scene for scene in scenes if scene["name"] == SCENE_NAME)
scene_contents = get_scene_contents(scene["token"], samples_all, 
                                    sample_data_all, ego_poses_all, calibrated_sensors_all,
                                    sample_annotations_all=sample_annotations_all,
                                    instances_all=instances_all)
samples = scene_contents["samples"]
sample_data = scene_contents["sample_data"]
ego_poses = scene_contents["ego_poses"]
calibrated_sensors = scene_contents["calibrated_sensors"]

# Create a sliding windows
window_ranges, used_ranges = create_sliding_windows(len(samples), window_size=WINDOW_SIZE, stride=STRIDE)

for window_count, ((i_start, i_end), (u_start, u_end)) in enumerate(zip(window_ranges, used_ranges)):
    # Get the contents of the current sliding window
    window_contents_cam = get_sample_window_contents(i_start, i_end, samples, sample_data, ego_poses, calibrated_sensors,
                                                     sensor_token=sensor_lookup[CAMERA_CHANNEL])
    
    window_samples = window_contents_cam["samples"]
    window_sample_data_cam = window_contents_cam["sample_data"]
    window_ego_poses_cam = window_contents_cam["ego_poses"]
    window_calibrated_sensors_cam = window_contents_cam["calibrated_sensors"]

    # Get the images for the sliding window
    images = [Image.open(NUSCENES_ROOT / sd["filename"]) for sd in window_sample_data_cam.values()]
    images = np.stack([np.array(img) for img in images], axis=0).astype(np.uint8)  # uint8 ndarray shape (N, H, W, 3)
    # Calculate the frame rate of the video based on the timestamps of the first and last sample_data in the sliding window
    timestamps = [sd["timestamp"] for sd in window_sample_data_cam.values()]
    target_fps = 1e6 / (timestamps[-1] - timestamps[0]) * (len(timestamps) - 1)  # fps = (N-1)

    # Inference depth using Video-Depth-Anything
    depths, fps = model.infer_video_depth(images, target_fps,  # depths: float32 ndarray shape (N, H, W), fps: float
                                          input_size=INPUT_SIZE, device=device, fp32=FP32)
    
    for used_idx in range(u_start, u_end):
        used_image = images[used_idx]
        used_metric_depth = depths[used_idx]
        used_sample_data = list(window_sample_data_cam.values())[used_idx]
        used_calibrated_sensor_cam = window_calibrated_sensors_cam[used_sample_data["calibrated_sensor_token"]]
        used_ego_pose_cam = window_ego_poses_cam[used_sample_data["ego_pose_token"]]

        # Generate a point cloud by pseudo-lidar from the depth map and transform to global coordinates
        pseudo_lidar_points = depth_map_to_point_cloud(used_metric_depth,
                                                       np.array(used_calibrated_sensor_cam["camera_intrinsic"]),
                                                       depth_threshold=80.0)
        pseudo_points_ego = transform_cam_to_ego(pseudo_lidar_points,
                                                 camera_translation=used_calibrated_sensor_cam["translation"],
                                                 camera_quaternion=used_calibrated_sensor_cam["rotation"])
        pseudo_points_global = transform_ego_to_global(pseudo_points_ego,
                                                       ego_translation=used_ego_pose_cam["translation"],
                                                       ego_quaternion=used_ego_pose_cam["rotation"])

        # Show the depth map and pseudo LiDAR in the first sliding window
        if window_count == 0 and used_idx < 3:
            plot_depth_with_original_image(used_metric_depth, np.array(used_image))

            ###### Ground truth LiDAR comparison ######
            sample_contents_lidar = get_sample_contents(i_start+used_idx, samples, sample_data, ego_poses, calibrated_sensors,
                                                        sensor_token=sensor_lookup[LIDAR_CHANNEL])
            # Read the point cloud
            lidar_path = NUSCENES_ROOT / list(sample_contents_lidar["sample_data"].values())[0]["filename"]
            lidar_points = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
            points_lidar = lidar_points[:, :3]  # Extract x, y, z coordinates
            intensity = lidar_points[:, 3]  # Extract intensity values
            # Transform the points from the LiDAR frame to the global frame
            calibrated_sensor_lidar = list(sample_contents_lidar["calibrated_sensors"].values())[0]
            lidar_points_ego = transform_lidar_to_ego(points_lidar, 
                                                      lidar_translation=calibrated_sensor_lidar["translation"],
                                                      lidar_quaternion=calibrated_sensor_lidar["rotation"])
            ego_pose_lidar = list(sample_contents_lidar["ego_poses"].values())[0]
            lidar_points_global = transform_ego_to_global(lidar_points_ego,
                                                            ego_translation=ego_pose_lidar["translation"],
                                                            ego_quaternion=ego_pose_lidar["rotation"])
        
            # Visualize the point cloud using Open3D
            fig = plot_pseudo_lidar_with_ground_truth(
                pseudo_lidar_points=pseudo_points_global,
                ground_truth_points=lidar_points_global,
                down_sample_size=0.3,
                axis_translation=ego_pose_lidar["translation"],
                axis_quaternion=ego_pose_lidar["rotation"]
            )
            fig.show()